# Day 10-11：Rerank 与混合检索

昨天学了分块策略和向量检索。但向量检索有一个天然缺陷：**Query 和 Document 独立编码，无法捕捉交互信息**。

什么意思？看这个例子：
- Query: "Python 怎么读文件"
- Doc1: "Python 文件操作指南" (向量相似度 0.82)
- Doc2: "用 open() 函数读取文件内容" (向量相似度 0.78)

向量检索会把 Doc1 排在前面，但 Doc2 才是真正回答问题的文档。

**Rerank 就是来解决这个问题的。**

## 一、Bi-Encoder vs Cross-Encoder

这是理解 Rerank 的关键概念。

### Bi-Encoder（向量检索用的编码器）

```
Query → [Encoder] → 向量Q ──┐
                              ├→ 余弦相似度 → 分数
Doc   → [Encoder] → 向量D ──┘
```

- Query 和 Doc **分别编码**，互不影响
- 优点：Doc 的向量可以预计算，检索时只需编码 Query，速度极快
- 缺点：无法捕捉 Query 和 Doc 之间的交互关系

### Cross-Encoder（Rerank 用的编码器）

```
[Query, Doc] → [Encoder] → 分数
```

- Query 和 Doc **拼接后一起编码**
- 模型内部可以做 Cross-Attention，捕捉每个词之间的关系
- 优点：精度远高于 Bi-Encoder
- 缺点：每个 (Query, Doc) 对都要过一次模型，速度慢

In [ ]:
# ===== Bi-Encoder vs Cross-Encoder 的核心区别 =====
# 用模拟数据直观展示两种方式的差异

import numpy as np

# 模拟场景：用户查询 "Python 读文件的方法"
query = "Python 读文件的方法"

documents = [
    "Python 文件操作完整指南",           # 相关但不直接回答
    "用 open() 函数读取文件内容",         # 最佳答案
    "Python 字符串处理方法",             # 不太相关
    "readlines() 读取所有行到列表",       # 直接回答
    "Java 文件读写操作",                 # 不相关
]

# ===== Bi-Encoder 模拟 =====
# 独立编码，用余弦相似度排序
np.random.seed(42)
dim = 8

# 模拟 Bi-Encoder 的打分（独立编码，无法理解交互）
bi_scores = [0.82, 0.78, 0.45, 0.71, 0.38]

print("Bi-Encoder 排序结果（向量检索）：")
print("=" * 55)
bi_ranked = sorted(enumerate(bi_scores), key=lambda x: x[1], reverse=True)
for rank, (idx, score) in enumerate(bi_ranked):
    print(f"  Rank {rank+1}: [{score:.2f}] {documents[idx]}")

# ===== Cross-Encoder 模拟 =====
# 拼接编码，能理解 Query-Doc 交互
cross_scores = [0.65, 0.92, 0.22, 0.88, 0.15]

print("\nCross-Encoder 排序结果（Rerank）：")
print("=" * 55)
cross_ranked = sorted(enumerate(cross_scores), key=lambda x: x[1], reverse=True)
for rank, (idx, score) in enumerate(cross_ranked):
    print(f"  Rank {rank+1}: [{score:.2f}] {documents[idx]}")

print("\n" + "=" * 55)
print("关键差异：")
print(f"  Bi-Encoder Top-1:   '{documents[bi_ranked[0][0]]}' (0.82)")
print(f"  Cross-Encoder Top-1: '{documents[cross_ranked[0][0]]}' (0.92)")
print("\nCross-Encoder 能理解'读文件'和'open()函数'的语义关系，")
print("而 Bi-Encoder 只能基于表面语义相似度打分。")

## 二、Two-Stage 检索架构

既然 Bi-Encoder 快但不准，Cross-Encoder 准但慢，那就组合使用：

```
Stage 1: Bi-Encoder（召回）
  百万文档 → 快速检索 → Top-100

Stage 2: Cross-Encoder（精排）
  Top-100 → 精确打分 → Top-5
```

这就是搜索引擎的经典架构：**粗排 → 精排**。

Google、Bing、百度都是这么做的——先用便宜的方式快速缩小范围，再用昂贵的方式精确排序。

In [ ]:
# ===== Two-Stage 检索的完整流程 =====

import numpy as np
from dataclasses import dataclass

@dataclass
class Document:
    content: str
    bi_score: float = 0.0
    cross_score: float = 0.0
    metadata: dict = None

class TwoStageRetriever:
    """
    Two-Stage 检索器
    
    Stage 1: Bi-Encoder 召回 Top-N
    Stage 2: Cross-Encoder 精排 Top-K
    """
    
    def __init__(self, recall_n=20, final_k=5):
        self.recall_n = recall_n  # Stage 1 召回数量
        self.final_k = final_k    # Stage 2 最终返回数量
        self.documents = []
    
    def add_documents(self, docs: list[Document]):
        """添加文档到索引"""
        self.documents = docs
        print(f"索引 {len(docs)} 个文档")
    
    def stage1_recall(self, query: str) -> list[Document]:
        """
        Stage 1: Bi-Encoder 召回
        用向量相似度快速筛选 Top-N
        """
        # 按 bi_score 排序，取 Top-N
        sorted_docs = sorted(self.documents, key=lambda d: d.bi_score, reverse=True)
        return sorted_docs[:self.recall_n]
    
    def stage2_rerank(self, query: str, candidates: list[Document]) -> list[Document]:
        """
        Stage 2: Cross-Encoder 精排
        用更精确的模型重新打分
        """
        # 按 cross_score 排序，取 Top-K
        sorted_docs = sorted(candidates, key=lambda d: d.cross_score, reverse=True)
        return sorted_docs[:self.final_k]
    
    def retrieve(self, query: str) -> list[Document]:
        """完整检索流程"""
        # Stage 1: 召回
        candidates = self.stage1_recall(query)
        print(f"Stage 1 召回: {len(self.documents)} → {len(candidates)} 个候选")
        
        # Stage 2: 精排
        results = self.stage2_rerank(query, candidates)
        print(f"Stage 2 精排: {len(candidates)} → {len(results)} 个结果")
        
        return results

# 模拟 100 个文档
np.random.seed(42)
sample_docs = []
for i in range(100):
    doc = Document(
        content=f"文档_{i}: 这是关于{'Python' if i < 30 else 'Java' if i < 60 else 'Go'}的第{i}篇技术文档",
        bi_score=np.random.uniform(0.3, 0.9),
        cross_score=np.random.uniform(0.1, 0.95),
    )
    sample_docs.append(doc)

# 让几个文档的 cross_score 特别高（模拟真正相关的文档）
sample_docs[5].cross_score = 0.95
sample_docs[5].content = "文档_5: Python open() 函数读取文件的完整教程"
sample_docs[12].cross_score = 0.91
sample_docs[12].content = "文档_12: Python readlines() 逐行读取文件"

# 但让它们的 bi_score 不是最高的（模拟向量检索不准的情况）
sample_docs[5].bi_score = 0.72
sample_docs[12].bi_score = 0.68

# 执行检索
retriever = TwoStageRetriever(recall_n=20, final_k=5)
retriever.add_documents(sample_docs)

print("\n" + "=" * 60)
print("Query: 'Python 怎么读文件'")
print("=" * 60)
results = retriever.retrieve("Python 怎么读文件")

print("\n最终 Top-5 结果：")
for i, doc in enumerate(results):
    print(f"  {i+1}. [{doc.cross_score:.2f}] {doc.content}")

print("\n" + "=" * 60)
print("\n对比：如果只用 Stage 1（纯向量检索）：")
bi_top5 = sorted(sample_docs, key=lambda d: d.bi_score, reverse=True)[:5]
for i, doc in enumerate(bi_top5):
    print(f"  {i+1}. [{doc.bi_score:.2f}] {doc.content}")

print("\n结论：Two-Stage 检索通过 Rerank 找到了真正相关的文档")

## 三、BM25 + 向量的混合检索

向量检索擅长语义匹配，但对精确关键词匹配反而不准。

比如用户搜 "Qwen2.5-7B"，向量检索可能返回关于"大语言模型"的文档，
而不是精确包含 "Qwen2.5-7B" 这个字符串的文档。

**BM25** 是经典的关键词检索算法，擅长精确匹配。把两者结合，取长补短。

### BM25 原理简介

BM25 基于**词频（TF）**和**逆文档频率（IDF）**：
- TF：一个词在文档中出现的次数越多，相关性越高
- IDF：一个词在越少的文档中出现，区分度越高
- "的" 出现在所有文档中 → IDF 低，不重要
- "PagedAttention" 只在少数文档中出现 → IDF 高，很重要

In [ ]:
# ===== BM25 算法的简化实现 =====

import math
from collections import Counter

class SimpleBM25:
    """
    BM25 的简化实现
    
    核心公式：
    score(q, d) = Σ IDF(t) * (TF(t,d) * (k1+1)) / (TF(t,d) + k1 * (1 - b + b * |d|/avgdl))
    
    简化理解：
    - 词出现越多 → 分数越高（但有饱和机制）
    - 词越稀有 → 权重越高（IDF）
    - 文档越短 → 匹配越精确（长度归一化）
    """
    
    def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1  # 词频饱和参数
        self.b = b    # 文档长度归一化参数
        self.docs = []
        self.doc_freqs = {}  # 每个词出现在几个文档中
        self.avg_dl = 0      # 平均文档长度
    
    def _tokenize(self, text):
        """简单分词：按空格和标点切分"""
        import re
        return re.findall(r'[\w]+', text.lower())
    
    def index(self, documents: list[str]):
        """建立索引"""
        self.docs = [self._tokenize(doc) for doc in documents]
        
        # 计算平均文档长度
        self.avg_dl = sum(len(d) for d in self.docs) / len(self.docs)
        
        # 计算每个词的文档频率
        df = Counter()
        for doc in self.docs:
            unique_terms = set(doc)
            for term in unique_terms:
                df[term] += 1
        self.doc_freqs = df
        
        print(f"索引 {len(self.docs)} 个文档，平均长度 {self.avg_dl:.1f} 词")
        print(f"词汇量: {len(self.doc_freqs)} 个不同词")
    
    def _idf(self, term):
        """计算 IDF：词越稀有，IDF 越高"""
        n = len(self.docs)
        df = self.doc_freqs.get(term, 0)
        return math.log((n - df + 0.5) / (df + 0.5) + 1)
    
    def score(self, query: str, doc_idx: int) -> float:
        """计算 Query 和指定文档的 BM25 分数"""
        query_terms = self._tokenize(query)
        doc = self.docs[doc_idx]
        doc_len = len(doc)
        tf_map = Counter(doc)
        
        total = 0.0
        for term in query_terms:
            tf = tf_map.get(term, 0)
            idf = self._idf(term)
            numerator = tf * (self.k1 + 1)
            denominator = tf + self.k1 * (1 - self.b + self.b * doc_len / self.avg_dl)
            total += idf * numerator / denominator
        
        return total
    
    def search(self, query: str, top_k: int = 5) -> list[tuple]:
        """检索 Top-K 文档"""
        scores = [(i, self.score(query, i)) for i in range(len(self.docs))]
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k]

# 演示
documents = [
    "LangGraph 是 LangChain 团队开发的状态图框架",
    "Python open() 函数可以读取文件内容",
    "Qwen2.5-7B 是阿里通义千问的最新模型",
    "大语言模型的应用越来越广泛",
    "BM25 是经典的关键词检索算法",
    "向量检索基于语义相似度匹配",
    "Qwen2.5 支持 Tool Calling 功能",
]

bm25 = SimpleBM25()
bm25.index(documents)

# 测试精确关键词匹配
print("\n" + "=" * 50)
print("Query: 'Qwen2.5-7B'")
results = bm25.search("Qwen2.5-7B")
for idx, score in results:
    print(f"  [{score:.3f}] {documents[idx]}")

print("\nQuery: 'Python 读文件'")
results = bm25.search("Python 读文件")
for idx, score in results:
    print(f"  [{score:.3f}] {documents[idx]}")

print("\n" + "=" * 50)
print("BM25 的优势：精确关键词匹配，如 'Qwen2.5-7B'")
print("BM25 的劣势：不理解语义，'读文件' 和 'open()函数' 无法关联")

## 四、分数融合策略

有了 BM25 和向量检索两路结果，怎么合并？

### 方法1：线性加权

```
final_score = α * bm25_score + (1-α) * vector_score
```

- α = 0.5：两者平等
- α = 0.7：更重视关键词匹配
- α = 0.3：更重视语义匹配

### 方法2：RRF（Reciprocal Rank Fusion）

```
score = Σ 1 / (k + rank_i)
```

其中 k 通常取 60。RRF 不关心具体分数，只关心排名。

**RRF 的优势**：不需要对分数做归一化，不同检索器的分数尺度可能完全不同。

In [ ]:
# ===== 两种融合策略的实现 =====

def linear_fusion(bm25_results, vector_results, alpha=0.5):
    """
    线性加权融合
    final_score = α * bm25_score + (1-α) * vector_score
    """
    # 归一化分数到 [0, 1]
    def normalize(results):
        if not results:
            return {}
        max_score = max(s for _, s in results)
        min_score = min(s for _, s in results)
        rng = max_score - min_score if max_score != min_score else 1
        return {idx: (score - min_score) / rng for idx, score in results}
    
    bm25_norm = normalize(bm25_results)
    vector_norm = normalize(vector_results)
    
    all_ids = set(bm25_norm.keys()) | set(vector_norm.keys())
    scores = {}
    for idx in all_ids:
        b_score = bm25_norm.get(idx, 0)
        v_score = vector_norm.get(idx, 0)
        scores[idx] = alpha * b_score + (1 - alpha) * v_score
    
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


def rrf_fusion(bm25_results, vector_results, k=60):
    """
    RRF（Reciprocal Rank Fusion）
    score = Σ 1 / (k + rank_i)
    """
    # 把结果转成排名
    def to_rank(results):
        return {idx: rank + 1 for rank, (idx, _) in enumerate(results)}
    
    bm25_ranks = to_rank(bm25_results)
    vector_ranks = to_rank(vector_results)
    
    all_ids = set(bm25_ranks.keys()) | set(vector_ranks.keys())
    scores = {}
    for idx in all_ids:
        score = 0
        if idx in bm25_ranks:
            score += 1 / (k + bm25_ranks[idx])
        if idx in vector_ranks:
            score += 1 / (k + vector_ranks[idx])
        scores[idx] = score
    
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


# 模拟数据
documents = [
    "LangGraph 状态图框架",
    "Python open() 读文件",
    "Qwen2.5-7B 模型介绍",
    "大语言模型应用",
    "BM25 关键词检索",
]

# 模拟两路检索结果：(doc_idx, score)
bm25_results = [(2, 5.2), (4, 3.1), (1, 2.8), (0, 1.5), (3, 0.8)]
vector_results = [(3, 0.92), (0, 0.85), (2, 0.78), (1, 0.72), (4, 0.45)]

print("BM25 排序：")
for idx, score in bm25_results:
    print(f"  [{score:.1f}] {documents[idx]}")

print("\n向量检索排序：")
for idx, score in vector_results:
    print(f"  [{score:.2f}] {documents[idx]}")

print("\n" + "=" * 50)
print("线性加权融合 (α=0.5)：")
print("=" * 50)
linear_results = linear_fusion(bm25_results, vector_results, alpha=0.5)
for idx, score in linear_results:
    print(f"  [{score:.3f}] {documents[idx]}")

print("\n" + "=" * 50)
print("RRF 融合 (k=60)：")
print("=" * 50)
rrf_results = rrf_fusion(bm25_results, vector_results, k=60)
for idx, score in rrf_results:
    print(f"  [{score:.5f}] {documents[idx]}")

print("\n" + "=" * 50)
print("观察：")
print("- BM25 把 Qwen2.5-7B 排第一（精确匹配 'Qwen2.5-7B'）")
print("- 向量检索把大语言模型排第一（语义最相关）")
print("- 融合后 Qwen2.5-7B 和大语言模型都排在前面")
print("- RRF 不需要归一化，实现更简单，工程中更常用")

## 五、常用 Rerank 模型

| 模型 | 来源 | 特点 | 推荐场景 |
|------|------|------|----------|
| bge-reranker-v2-m3 | BAAI | 多语言，中文效果好，开源免费 | 中文 RAG 首选 |
| Cohere Rerank | Cohere | API 服务，效果好 | 不想部署模型时 |
| jina-reranker-v2 | Jina | 多语言，开源 | 多语言场景 |
| ms-marco-MiniLM | Microsoft | 轻量，英文为主 | 英文场景 |

中文项目推荐 **bge-reranker-v2-m3**，开源免费，中文效果最好。

In [ ]:
# ===== 使用 HuggingFace 的 Reranker =====
# 注意：实际运行需要安装 transformers 和 torch
# pip install transformers torch

# 以下代码展示 Reranker 的使用方式
# 由于环境限制，这里只展示代码框架

def rerank_with_model(query, documents, model_name="BAAI/bge-reranker-v2-m3"):
    """
    使用 Cross-Encoder 模型进行 Rerank
    
    实际代码：
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    
    scores = []
    for doc in documents:
        inputs = tokenizer(query, doc, return_tensors='pt', truncation=True, max_length=512)
        score = model(**inputs).logits.item()
        scores.append(score)
    
    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
    return [(documents[idx], score) for idx, score in ranked]
    """
    pass

# 用 API 方式调用（以 Cohere 为例）
def rerank_with_api(query, documents, api_key=None):
    """
    使用 Cohere Rerank API
    
    实际代码：
    import cohere
    
    co = cohere.Client(api_key)
    results = co.rerank(
        query=query,
        documents=documents,
        top_n=5,
        model='rerank-v3.5'
    )
    
    return [(documents[r.index], r.relevance_score) for r in results.results]
    """
    pass

print("Rerank 模型使用方式：")
print("=" * 50)
print("\n方式1: 本地部署（推荐 bge-reranker-v2-m3）")
print("  pip install transformers torch")
print("  模型约 1.1GB，需要 GPU 加速")
print("\n方式2: API 调用（推荐 Cohere Rerank）")
print("  pip install cohere")
print("  按调用次数付费，无需 GPU")
print("\n方式3: 用 LLM 做 Rerank（不推荐）")
print("  让 LLM 判断文档相关性，太慢太贵")
print("  专用 Reranker 模型比 LLM 快 100 倍")

## 六、完整混合检索 + Rerank 流程

把所有组件组合起来：

```
用户 Query
    ↓
┌─────────────────────────────────────┐
│  并行检索                            │
│  ├─ BM25 检索 → Top-50              │
│  └─ 向量检索 → Top-50               │
└─────────────────────────────────────┘
    ↓
┌─────────────────────────────────────┐
│  RRF 融合 → 去重 → Top-20           │
└─────────────────────────────────────┘
    ↓
┌─────────────────────────────────────┐
│  Cross-Encoder Rerank → Top-5       │
└─────────────────────────────────────┘
    ↓
┌─────────────────────────────────────┐
│  LLM 拿到 Top-5 Context 生成答案    │
└─────────────────────────────────────┘
```

In [ ]:
# ===== 完整混合检索 + Rerank 系统 =====

import numpy as np
from collections import Counter


class HybridRetriever:
    """
    混合检索系统：BM25 + 向量 + Rerank
    
    完整流程：
    1. BM25 检索 Top-N
    2. 向量检索 Top-N
    3. RRF 融合
    4. Cross-Encoder Rerank
    5. 返回 Top-K
    """
    
    def __init__(self, bm25_top_n=20, vector_top_n=20, final_k=5):
        self.bm25_top_n = bm25_top_n
        self.vector_top_n = vector_top_n
        self.final_k = final_k
        self.documents = []
        self.bm25 = None
        self.doc_embeddings = None
    
    def index(self, documents: list[str]):
        """索引文档"""
        self.documents = documents
        
        # 建立 BM25 索引
        self.bm25 = SimpleBM25()
        self.bm25.index(documents)
        
        # 计算向量 Embedding（模拟）
        dim = 16
        self.doc_embeddings = np.random.randn(len(documents), dim)
        
        print(f"索引完成: {len(documents)} 个文档")
    
    def _vector_search(self, query: str, top_n: int) -> list[tuple]:
        """向量检索（模拟）"""
        np.random.seed(hash(query) % 2**31)
        query_emb = np.random.randn(self.doc_embeddings.shape[1])
        
        scores = []
        for i, doc_emb in enumerate(self.doc_embeddings):
            sim = np.dot(query_emb, doc_emb) / (
                np.linalg.norm(query_emb) * np.linalg.norm(doc_emb)
            )
            scores.append((i, sim))
        
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_n]
    
    def _rrf_fusion(self, bm25_results, vector_results, k=60):
        """RRF 融合"""
        bm25_ranks = {idx: r + 1 for r, (idx, _) in enumerate(bm25_results)}
        vector_ranks = {idx: r + 1 for r, (idx, _) in enumerate(vector_results)}
        
        all_ids = set(bm25_ranks.keys()) | set(vector_ranks.keys())
        scores = {}
        for idx in all_ids:
            s = 0
            if idx in bm25_ranks:
                s += 1 / (k + bm25_ranks[idx])
            if idx in vector_ranks:
                s += 1 / (k + vector_ranks[idx])
            scores[idx] = s
        
        return sorted(scores.items(), key=lambda x: x[1], reverse=True)
    
    def _mock_rerank(self, query: str, candidates: list[tuple]) -> list[tuple]:
        """模拟 Rerank（实际用 bge-reranker）"""
        # 模拟 Cross-Encoder 打分
        np.random.seed(hash(query + "rerank") % 2**31)
        reranked = []
        for idx, _ in candidates:
            score = np.random.uniform(0.3, 0.95)
            reranked.append((idx, score))
        reranked.sort(key=lambda x: x[1], reverse=True)
        return reranked
    
    def retrieve(self, query: str) -> list[tuple[str, float]]:
        """完整检索流程"""
        # Stage 1a: BM25
        bm25_results = self.bm25.search(query, self.bm25_top_n)
        print(f"BM25 召回: {len(bm25_results)} 个")
        
        # Stage 1b: 向量检索
        vector_results = self._vector_search(query, self.vector_top_n)
        print(f"向量召回: {len(vector_results)} 个")
        
        # Stage 2: RRF 融合
        fused = self._rrf_fusion(bm25_results, vector_results)
        print(f"RRF 融合后: {len(fused)} 个")
        
        # Stage 3: Rerank
        reranked = self._mock_rerank(query, fused[:20])
        print(f"Rerank 后: Top-{self.final_k}")
        
        return [(self.documents[idx], score) for idx, score in reranked[:self.final_k]]

# 使用示例
docs = [
    "LangGraph 是 LangChain 的状态图框架，用于构建 Agent 工作流",
    "Python 的 open() 函数用于打开文件，read() 读取内容",
    "Qwen2.5-7B 是阿里通义千问的 7B 参数模型，支持 Tool Calling",
    "大语言模型在自然语言处理领域有广泛应用",
    "BM25 基于词频和逆文档频率进行关键词匹配",
    "向量检索通过 Embedding 将文本映射到高维空间",
    "vLLM 使用 PagedAttention 优化推理显存管理",
    "LoRA 通过低秩分解实现参数高效微调",
    "RAG 系统的核心是检索质量，分块策略很关键",
    "Cross-Encoder 将 Query 和 Doc 拼接编码，精度更高",
]

retriever = HybridRetriever()
retriever.index(docs)

print("\n" + "=" * 50)
print("Query: 'Qwen2.5 模型怎么用'")
print("=" * 50)
results = retriever.retrieve("Qwen2.5 模型怎么用")
print("\n最终结果：")
for i, (doc, score) in enumerate(results):
    print(f"  {i+1}. [{score:.3f}] {doc}")

## 今日总结

今天掌握了 RAG 系统中最关键的检索优化技术。

### 核心知识

| 技术 | 解决什么问题 | 关键点 |
|------|------------|--------|
| Cross-Encoder | 向量检索精度不够 | Query+Doc 拼接编码，捕捉交互 |
| Two-Stage 检索 | 精度和速度的平衡 | 粗排（Bi-Encoder）→ 精排（Cross-Encoder）|
| BM25 | 精确关键词匹配 | TF-IDF 思想，Qwen2.5-7B 这类词必须精确匹配 |
| RRF 融合 | 合并多路检索结果 | 基于排名，不需要分数归一化 |
| bge-reranker | 中文 Rerank | 开源免费，中文效果最好 |

**明天学什么：** vLLM 推理加速原理——KV Cache、PagedAttention、Continuous Batching。

**写在简历上的要点：**
"熟悉 RAG 检索优化，包括 Two-Stage 检索架构（Bi-Encoder 召回 + Cross-Encoder 精排）、BM25+向量混合检索、RRF 分数融合策略。"